In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ultralytics

In [ ]:
!nvidia-smi

In [ ]:
from ultralytics import YOLO

In [ ]:
# Checking and Analysing the GPU.

In [ ]:
import torch
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
import torch
print(torch.cuda.device_count())

In [ ]:
# Preprocessed Data Importing in kaggle file.

In [1]:
import os
import shutil
import glob

# Source input directories
src1 = "/kaggle/input/datasets/dixitshashank937/aircraft-dataset/military-20260317T092554Z-3-001/military"
src2 = "/kaggle/input/datasets/dixitshashank937/aircraft-dataset/military-20260317T092554Z-3-002/military"

# Destination YOLO dataset in working directory.
dst = "/kaggle/working/dataset"
os.makedirs(dst, exist_ok=True)

# Create YOLO folder structure
for folder in ["images/train", "images/val", "images/test", 
               "labels/train", "labels/val", "labels/test"]:
    os.makedirs(os.path.join(dst, folder), exist_ok=True)

# Function to copy images and labels from a source dataset into YOLO folders
def copy_to_yolo(src_folder, dst_folder):
    for split in ["train", "val", "test"]:
        # Copy images
        img_src = os.path.join(src_folder, f"images/{split}")
        img_dst = os.path.join(dst_folder, f"images/{split}")
        if os.path.exists(img_src):
            for img_file in glob.glob(os.path.join(img_src, "*")):
                shutil.copy(img_file, img_dst)
        
        # Copy labels
        lbl_src = os.path.join(src_folder, f"labels/{split}")
        lbl_dst = os.path.join(dst_folder, f"labels/{split}")
        if os.path.exists(lbl_src):
            for lbl_file in glob.glob(os.path.join(lbl_src, "*")):
                shutil.copy(lbl_file, lbl_dst)

# Copy both datasets into one YOLO structure
copy_to_yolo(src1, dst)
copy_to_yolo(src2, dst)

print("Both datasets combined into single YOLO folder structure in:", dst)

Both datasets combined into single YOLO folder structure in: /kaggle/working/dataset


In [ ]:
# Creating YOLO data.yaml created at: /kaggle/working/dataset/data.yaml

In [3]:
import os

# Base path in Kaggle working directory
base_path = "/kaggle/working/dataset"

# Number of classes
num_classes = 43

# Class names
class_names = [
    "A10", "A400M", "AG600", "AV8B", "B1", "B2", "B52", "Be200", "C130", "C17",
    "C2", "C5", "E2", "E7", "EF2000", "F117", "F14", "F15", "F16", "F18",
    "F22", "F35", "F4", "J20", "JAS39", "MQ9", "Mig31", "Mirage2000", "P3", "RQ4",
    "Rafale", "SR71", "Su34", "Su57", "Tornado", "Tu160", "Tu95", "U2", "US2",
    "V22", "Vulcan", "XB70", "YF23"
]

# YAML content
data_yaml = f"""
train: {os.path.join(base_path, 'images/train')}
val: {os.path.join(base_path, 'images/val')}
test: {os.path.join(base_path, 'images/test')}

nc: {num_classes}
names: {class_names}
"""

# Save YAML file
yaml_path = os.path.join(base_path, "data.yaml")
os.makedirs(base_path, exist_ok=True)
with open(yaml_path, "w") as f:
    f.write(data_yaml)

print(f"YOLO data.yaml created at: {yaml_path}")

YOLO data.yaml created at: /kaggle/working/dataset/data.yaml


In [ ]:
# Copy last weights of Collab in working directory.

In [2]:
import shutil
import os

# Source and destination
src = "/kaggle/input/datasets/dixitshashank937/11111wei/last 10(1).pt"
dst = "/kaggle/working/last.pt"

# Copy the file
shutil.copy(src, dst)

print(f"last.pt copied to working directory: {dst}")

last.pt copied to working directory: /kaggle/working/last.pt


In [ ]:
# Model Training.

In [4]:
!pip install ultralytics --upgrade

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.8 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/datasets/dixitshashank937/11111wei/last 10(1).pt")  # or last.pt

metrics = model.val(
    data="/kaggle/working/dataset/data.yaml",
    split="val",
    save=True,
    plots=True
)

In [ ]:
model = YOLO("/kaggle/working/last.pt")  # previous checkpoint

model.train(
    data="/kaggle/working/dataset/data.yaml",
    epochs=300,       # new training epochs.
    batch=16,
    imgsz=640,
    device=0,
    augment=True,
    save=True,
    lr0=0.001,        # learning rate for fine-tuning.
    patience=20,
    save_period=10,
)

In [ ]:
import shutil

src = "/kaggle/input/datasets/dixitshashank937/results"
dst = "/kaggle/working/train"

shutil.copytree(src, dst)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for name in files:
        print(os.path.join(root, name))

In [ ]:
import cv2
from ultralytics import YOLO

# Load your trained model
model = YOLO("/kaggle/working/last.pt")  # change path if needed

# Input video path
video_path = "/kaggle/input/datasets/dixitshashank937/vvvvvvvv/137859-767696316_medium.mp4"  # change this

# Output video path
output_path = "/kaggle/working/output.mp4"

# Open video
cap = cv2.VideoCapture(video_path)

# Get video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break

    # Run YOLO inference
    results = model(frame)

    # Draw results on frame
    annotated_frame = results[0].plot()

    # Write frame to output video
    out.write(annotated_frame)

    # Show live (optional)
    cv2.imshow("Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("✅ Video processing complete! Saved at:", output_path)


0: 384x640 1 AV8B, 781.4ms
Speed: 16.0ms preprocess, 781.4ms inference, 47.2ms postprocess per image at shape (1, 3, 384, 640)


In [1]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/last.pt")

model.predict(
    source="/kaggle/input/datasets/dixitshashank937/vvvvvvvv/137859-767696316_medium.mp4",
    save=True,
    imgsz=640,
    conf=0.3,
    stream=True  # IMPORTANT
)

<generator object BasePredictor.stream_inference at 0x7a3276787e20>